[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nrao/astrohack/blob/v0.10.1/docs/beamcut_tutorial.ipynb)

![astrohack](astrohack_logo.png)

# Beam cut analysis tutorial

Beam cuts are a commonly used measurement to infer properties of the telescope beam without having to resort to a much more expensive full holography. With that in mind `astrohack.beamcut` was created with an intent to aid VLA operations and in the near future the commissioning of the ngVLA prototype antenna. In this tutorial we go through the astrohack reduction of a calibrated ms.

In [ ]:
import os

try:
    import astrohack

    print("AstroHACK version", astrohack.__version__, "already installed.")
except ImportError as e:
    print(e)
    print("Installing AstroHACK")

    os.system("pip install astrohack")

    import astrohack

    print("astrohack version", astrohack.__version__, " installed.")

## Download tutorial data

In [ ]:
# Download data.
import toolviper

basename = "kband_beamcut_small"
ms_name = f"data/{basename}.ms"

toolviper.utils.data.update()
toolviper.utils.data.download(file=f"{basename}.ms", folder="data")

## Starting local Dask client

In [ ]:
from toolviper.dask.client import local_client

parallel = False

if parallel:
    client = local_client(cores=4, memory_limit="4GB")
    print(client)
else:
    client = None

## Extract beam cut data to Astrohack formats

In this step we use `astrohack.extract_pointing` and `astrohack.extract_holog` to create an astrohack holog.zarr holography file from the ms. The usage of these functions originally created for holography purposes is needed to simplify the process of creating the beamcuts as `astrohack.extract_pointing` determines which antennas were moving and `astrohack.extract_holog` matches visibilities to pointing data in a convenient way.

In [ ]:
from astrohack import extract_pointing, extract_holog

point_name = f"data/{basename}.point.zarr"
holog_name = f"data/{basename}.holog.zarr"

# Extract pointing data to an astrohack file format
point_mds = extract_pointing(
    ms_name,
    point_name,
    overwrite=True,
    parallel=parallel
)

# Extract visibilities to an astrohack file format
holog_mds = extract_holog(
    ms_name,
    point_name,
    holog_name,
    data_column="DATA",  # This applies to this dataset only as it has been split
    overwrite=True,
    parallel=parallel,
)

## Running beamcut

`astrohack.beamcut` executes the following steps for each antenna & DDI combination:
1. Break up visibility data onto different cuts based on scans
2. Determine the cut direction based on the L and M distributions in each cut
3. Flatten the cut onto a distance from pointing center axis
4. Fit multiple gaussians to the multiple lobes present in the beam
5. Identify the primary beam and the first sidelobes
6. Measure primary beam offset and the first sidelobe ratio
7. If a destination is given, plots of the beam cut in the sky, the beam in amplitude and the beam in attenuation are produced along with a table with the properties of all beam lobes as measured with the gaussian fit.

The measurements in step 6 are indeed the crux of the matter, as the offset of the primary beam is an important measure of antenna optical alignment and the first sidelobe ratio is linked to focus offsets.


In [ ]:
from astrohack import beamcut

beamcut_name = f"data/{basename}.beamcut.zarr"

beamcut_mds = beamcut(
    holog_name,
    beamcut_name,
    destination=None,  # This parameter is to be filled with a destination directory for beamcut products.
    overwrite=True,
    parallel=parallel,
)

## Interact with a beamcut object

The beamcut object is the return of `astrohack.beamcut` and can also be obtained by opening a beamcut.zarr file on disk by using `astrohack.open_beamcut`.
This object allows the user to directly manipulate the data contained in the related file on disk, as well as requesting new plots and reports to be generated from the data on disk.

In [ ]:
from astrohack import open_beamcut

beamcut_mds = open_beamcut(beamcut_name)

beamcut_mds.summary()

### Observation summary

The observation summary contains relevant observation information for each antenna and DDI combination as well as cut timing and direction.

In [ ]:
beamcut_mds.observation_summary("data/beamcut_summary.txt")

### Interacting with datatree

Below its is shown how to interact directly with the data tree contained in the beamcut_mds by simply especifying the keys of interest to arrive at the desired xarray dataset.

In [ ]:
ea17_ddi_0 = beamcut_mds["ant_ea17"]["ddi_0"]["cut_0"]

ea17_ddi_0

### Creating beam cut  plots

Creating plots from the data in a beamcut_mds can be done by using the plotting methods.

#### Beamcut Sky coverage
The first of these methods is `plot_beam_cuts_over_sky` which creates a plot of all pointings for each antenna and DDI combination, colored by cut (scan).

In [ ]:
beamcut_exports = "beamcut_exports"

beamcut_mds.plot_beam_cuts_over_sky(
    beamcut_exports,
    ant="ea17",
    ddi=0,
    display=True,
    parallel=parallel,
)

#### Amplitude plots

The next plotting method is `plot_beamcut_in_amplitude` which plots the beamcut data in amplitude for each correlation and cut while identifying the lobes marked in the report.
Key properties are also shown, the primary beam offset, FWHM and the first side lobe ratio (FSLR).

In [ ]:
beamcut_mds.plot_beamcut_in_amplitude(
    beamcut_exports,
    ant="ea17",
    ddi=0,
    display=True,
    parallel=parallel,
)

#### Attenuation plots

The remaining plotting method is `plot_beamcut_in_attenuation` which plots each cut in attenuation in a way that let us compare the parallel hands directly.
Also present in this plot are the primary beam offset, FWHM and the first side lobe ratio (FSLR), however the numbers shown here are the average between the 2 parallel hands.

In [ ]:
beamcut_mds.plot_beamcut_in_attenuation(
    beamcut_exports,
    ant="ea17",
    ddi=0,
    display=True,
    parallel=parallel,
)

#### Create report on beam cut gaussian fit

Lastly, `create_beam_fit_report` produces a report with the fitted parameters to each of the lobes present in the beam.

In [ ]:
beamcut_mds.create_beam_fit_report(beamcut_exports, ant="ea17", ddi=0, parallel=False)

with open("beamcut_exports/beamcut_report_ant_ea17_ddi_0.txt", "r") as infile:
    for line in infile:
        print(line[:-1])

In [ ]:
if parallel:
    client.close()